In [ ]:
# Nhập thư viện cần thiết
import json
import logging
import os
import time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

import requests
import pytest


In [ ]:
BASE_URL = 'http://127.0.0.1:5000'
REVERSE_ENDPOINT = '/api/geocode/reverse'
HEADERS = {'Accept': 'application/json'}
TIMEOUT_SECONDS = 10

TEST_COORDINATES = [
    {'label': 'Hà Nội', 'lat': 21.0278, 'lon': 105.8342, 'expect_success': True},
    {'label': 'TP.HCM', 'lat': 10.8231, 'lon': 106.6297, 'expect_success': True},
    {'label': 'Invalid Lat', 'lat': 'a', 'lon': 105.8342, 'expect_success': False},
    {'label': 'Invalid Lon', 'lat': 21.0278, 'lon': 'b', 'expect_success': False},
    {'label': 'Missing', 'lat': None, 'lon': None, 'expect_success': False},
]


In [ ]:
def call_reverse_geocode(lat: Any, lon: Any, retries: int = 3) -> Dict[str, Any]:
    url = BASE_URL + REVERSE_ENDPOINT
    params = {'lat': lat, 'lon': lon}
    backoff = 1.0

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=HEADERS, params=params, timeout=TIMEOUT_SECONDS)
            if response.status_code == 429:
                time.sleep(backoff)
                backoff *= 2
                continue
            response.raise_for_status()
            if 'application/json' not in response.headers.get('Content-Type', ''):
                raise ValueError('Response is not JSON')
            return response.json()
        except requests.RequestException:
            if attempt == retries:
                raise
            time.sleep(backoff)
            backoff *= 2
    raise RuntimeError('Exceeded retry attempts')

In [ ]:
def validate_response_schema(resp_json: Dict[str, Any]) -> List[str]:
    errors: List[str] = []
    if not isinstance(resp_json, dict):
        return ['Response is not a JSON object']

    if 'ok' not in resp_json:
        errors.append('Missing field: ok')
    elif not isinstance(resp_json['ok'], bool):
        errors.append('Field ok must be boolean')

    if 'result' not in resp_json:
        errors.append('Missing field: result')
    elif resp_json['result'] is not None and not isinstance(resp_json['result'], dict):
        errors.append('Field result must be object or null')
    else:
        result = resp_json['result']
        if isinstance(result, dict):
            address = result.get('display_name')
            if address is None and 'address' not in result:
                errors.append('Missing address information')
            if 'lat' in result and not isinstance(result['lat'], (str, float, int)):
                errors.append('Field result.lat must be numeric or string')
            if 'lon' in result and not isinstance(result['lon'], (str, float, int)):
                errors.append('Field result.lon must be numeric or string')
    return errors


def check_response_status_and_json(response: requests.Response) -> Dict[str, Any]:
    if response.status_code != 200:
        raise ValueError(f'Unexpected status code: {response.status_code}')
    content_type = response.headers.get('Content-Type', '')
    if 'application/json' not in content_type:
        raise ValueError(f'Unexpected Content-Type: {content_type}')
    try:
        return response.json()
    except ValueError as exc:
        raise ValueError('Invalid JSON body') from exc


In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

FAILURE_DIR = os.path.join(os.path.dirname(__file__), 'failures')
if not os.path.exists(FAILURE_DIR):
    os.makedirs(FAILURE_DIR, exist_ok=True)


def save_failure(entry: Dict[str, Any], response: Dict[str, Any]) -> None:
    filename = os.path.join(FAILURE_DIR, f"failure_{int(time.time())}.json")
    payload = {'entry': entry, 'response': response}
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    logging.error('Saved failure data to %s', filename)


@pytest.mark.parametrize('entry', TEST_COORDINATES)
def test_reverse_geocode_and_save_failure(entry):
    lat = entry['lat']
    lon = entry['lon']
    try:
        result = call_reverse_geocode(lat, lon)
        errors = validate_response_schema(result)
        if errors:
            save_failure(entry, result)
        assert not errors, f'Validation errors: {errors}'
    except Exception as exc:
        save_failure(entry, {'error': str(exc)})
        raise


## Ghi log và lưu phản hồi lỗi
Cấu hình logging; khi validate thất bại lưu phản hồi vào ./failures.

## Chạy notebook và kiểm tra output trong VSCode
Hướng dẫn chạy toàn bộ suite kiểm tra và mở file lưu lỗi để debug.

1. Chạy từng cell để kiểm tra setup và API reverse-geocode.
2. Nếu cell cuối cùng không báo lỗi, API trả về dữ liệu hợp lệ.
3. Nếu có thất bại, mở thư mục `failures` và xem file JSON được lưu.
4. Bạn cũng có thể chạy `pytest reverse_geocode_test.ipynb` nếu notebook runner được cấu hình, hoặc chuyển code vào `tests/` để chạy bằng `pytest`.

In [ ]:
# Kiểm tra API reverse geocode trực tiếp
try:
    response = call_reverse_geocode(21.0278, 105.8342)
    print('HTTP OK, response JSON:')
    print(json.dumps(response, ensure_ascii=False, indent=2))
except Exception as exc:
    print('Reverse geocode call failed:', exc)


# Kiểm tra API Reverse Geocode
Notebook để kiểm tra xem API reverse-geocode có trả về dữ liệu hợp lệ hay không.